# MIMIC-IV Cohort Builder

Local DuckDB pipeline. No BigQuery.

In [ ]:
# Section 1: Setup
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])

import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('outputs', exist_ok=True)

con = duckdb.connect()
print('DuckDB connected. Reading from local CSV files.')

## Section 2: Confirm Column Names

In [ ]:
print('admissions.csv columns:')
print(con.execute("DESCRIBE SELECT * FROM 'data/admissions.csv' LIMIT 1").df())

print('\npatients.csv columns:')
print(con.execute("DESCRIBE SELECT * FROM 'data/patients.csv' LIMIT 1").df())

## Section 3: Count Distinct subject_id

In [ ]:
adm_count = con.execute("SELECT COUNT(DISTINCT subject_id) FROM 'data/admissions.csv'").fetchone()[0]
print(f'Total unique patients in admissions: {adm_count}')

pat_count = con.execute("SELECT COUNT(DISTINCT subject_id) FROM 'data/patients.csv'").fetchone()[0]
print(f'Total unique patients in patients table: {pat_count}')

## Section 4: Age and Gender Profile

Uses column names confirmed in Section 2.
Age at first admission = anchor_age + (year(admittime) - anchor_year).

In [ ]:
age_query = """
WITH first_adm AS (
    SELECT subject_id, MIN(admittime) AS admittime
    FROM 'data/admissions.csv'
    GROUP BY subject_id
),
base_age AS (
    SELECT
        p.subject_id,
        p.gender,
        p.anchor_age + (EXTRACT(year FROM a.admittime::TIMESTAMP) - p.anchor_year) AS age
    FROM 'data/patients.csv' p
    JOIN first_adm a ON p.subject_id = a.subject_id
)
SELECT
    subject_id,
    gender,
    age,
    CASE
        WHEN age < 30 THEN 'under_30'
        WHEN age < 50 THEN '30_to_50'
        WHEN age < 65 THEN '50_to_65'
        ELSE '65_plus'
    END AS age_group
FROM base_age
"""

df_age = con.execute(age_query).df()

age_order = ['under_30', '30_to_50', '50_to_65', '65_plus']
df_age['age_group'] = pd.Categorical(df_age['age_group'], categories=age_order, ordered=True)

crosstab_n = pd.crosstab(df_age['age_group'], df_age['gender'])
crosstab_pct = crosstab_n.div(crosstab_n.sum(axis=1), axis=0).mul(100).round(1)
print('Counts:')
print(crosstab_n)
print('\nRow percentages:')
print(crosstab_pct)

plot_df = df_age.groupby(['age_group', 'gender'], observed=True).size().reset_index(name='count')
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot_df, x='age_group', y='count', hue='gender', ax=ax)
ax.set_title('Patient Distribution by Age Group and Gender')
ax.set_xlabel('Age Group')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('outputs/age_gender_distribution.png', dpi=150)
plt.show()
print('Chart saved to outputs/age_gender_distribution.png')

## Section 5: Stratified 10k Sample

In [ ]:
sample_query = """
WITH first_adm AS (
    SELECT subject_id, MIN(admittime) AS admittime
    FROM 'data/admissions.csv'
    GROUP BY subject_id
),
base_age AS (
    SELECT
        p.subject_id,
        p.gender,
        p.anchor_age + (EXTRACT(year FROM a.admittime::TIMESTAMP) - p.anchor_year) AS age
    FROM 'data/patients.csv' p
    JOIN first_adm a ON p.subject_id = a.subject_id
),
base AS (
    SELECT
        subject_id,
        gender,
        age,
        CASE
            WHEN age < 30 THEN 'under_30'
            WHEN age < 50 THEN '30_to_50'
            WHEN age < 65 THEN '50_to_65'
            ELSE '65_plus'
        END AS age_group
    FROM base_age
),
strata_counts AS (
    SELECT age_group, gender, COUNT(*) AS available
    FROM base
    GROUP BY age_group, gender
),
total AS (
    SELECT SUM(available) AS total_patients FROM strata_counts
),
strata_targets AS (
    SELECT
        s.age_group,
        s.gender,
        s.available,
        FLOOR(s.available * 10000.0 / t.total_patients)::INTEGER AS target
    FROM strata_counts s CROSS JOIN total t
),
ranked AS (
    SELECT
        b.subject_id,
        b.age_group,
        b.gender,
        st.available,
        st.target,
        ROW_NUMBER() OVER (PARTITION BY b.age_group, b.gender ORDER BY random()) AS rn
    FROM base b
    JOIN strata_targets st ON b.age_group = st.age_group AND b.gender = st.gender
)
SELECT subject_id, age_group, gender, available, target
FROM ranked
WHERE rn <= target
"""

df_sample = con.execute(sample_query).df()

summary = (
    df_sample.groupby(['age_group', 'gender'])
    .agg(available=('available', 'first'), sampled=('subject_id', 'count'))
    .reset_index()
)
summary['pct_of_10k'] = (summary['sampled'] / 10000 * 100).round(2)
summary = summary[['age_group', 'gender', 'available', 'sampled', 'pct_of_10k']]
print(summary.to_string(index=False))
print(f'\nFinal cohort size: {len(df_sample)} patients')

## Section 6: Save the Cohort

In [ ]:
cohort = df_sample[['subject_id', 'age_group', 'gender']].copy()
cohort.to_csv('outputs/cohort_10k.csv', index=False)
print(f'Cohort saved. Shape: {cohort.shape[0]} rows, 3 columns.')
print(cohort.head())